# 05 Prior Authorization Risk Scoring Demo

Purpose: create a CMS-aligned PA risk prioritization prototype. Public CMS PA artifacts provide reporting schema and policy timing rules, not mature request-level labels. Therefore this notebook uses a seeded demo dataset plus transparent rule features.

In [1]:
from __future__ import annotations

import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, CPSC_DIR, PMPM_PROXY_REVENUE

for path in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

Project root: D:\Project 1\rcm-cms-mvp


## Run PA Demo Model

Why: the model demonstrates feature engineering and classifier workflow while staying honest about public-data limits.

In [2]:
from src.models.pa_classifier import run_pa_model

pa_predictions, pa_metrics = run_pa_model()
display(pa_metrics)
display(pa_predictions.sort_values("hybrid_denial_risk", ascending=False).head(20))

,model,accuracy,roc_auc,precision_approved,recall_approved,f1_approved,test_rows,note
0,gradient_boosting_pa_demo,0.933333,0.959135,0.961538,0.961538,0.961538,60,"Seeded public-data demo because CMS public PA files provide reporting schema, not request-level labels."


,procedure_type,diagnosis_category,payer_type,historical_approval_rate,doc_score,step_therapy_flag,is_expedited,prior_denial_history,missing_clinical_docs,approved,rule_denial_risk,rule_risk_bucket,ml_approval_probability,ml_denial_probability,hybrid_denial_risk,hybrid_risk_bucket
234,part_b_drug,oncology,MA_HMO,0.750629,0.535962,1,1,1,1,0,0.527174,high,0.003063,0.996937,0.738568,high
141,DME,respiratory,MA_HMO,0.567154,0.504327,1,0,0,1,0,0.507967,high,0.001227,0.998773,0.728830,high
166,SNF,geriatrics,MA_PPO,0.508452,0.544479,0,1,1,1,0,0.496220,high,0.004974,0.995026,0.720683,high
118,SNF,geriatrics,MA_PPO,0.477514,0.382683,0,0,0,1,0,0.464605,high,0.001872,0.998128,0.704691,high
45,DME,respiratory,MA_HMO,0.617009,0.526416,0,0,1,1,0,0.461291,high,0.003266,0.996734,0.702240,high
59,post_acute,rehab,MAPD,0.409462,0.613055,0,0,1,0,1,0.459835,high,0.003168,0.996832,0.701484,high
71,home_health,chronic_care,MAPD,0.648198,0.524598,0,1,1,1,0,0.450076,high,0.003422,0.996578,0.696001,high
100,surgery,cardiology,MA_PPO,0.766520,0.146385,0,0,0,1,1,0.437488,high,0.002245,0.997755,0.689608,high
157,DME,respiratory,MA_PPO,0.553406,0.643475,1,1,0,0,0,0.414489,medium,0.962931,0.037069,0.244650,low
215,home_health,chronic_care,MAPD,0.548151,0.606500,0,0,1,0,1,0.409428,medium,0.979357,0.020643,0.234474,low


## PA Risk Visuals

Elements: histograms show risk distribution; bars show high-risk groups by procedure/payer.

In [3]:
fig = px.histogram(pa_predictions, x="hybrid_denial_risk", color="hybrid_risk_bucket", title="PA Hybrid Denial Risk Distribution")
fig.show()

risk_by_proc = (
    pa_predictions.groupby(["procedure_type", "payer_type"], as_index=False)
    .agg(avg_hybrid_denial_risk=("hybrid_denial_risk", "mean"), high_risk_cases=("hybrid_risk_bucket", lambda s: int((s == "high").sum())))
    .sort_values("avg_hybrid_denial_risk", ascending=False)
)
risk_by_proc.to_csv(TABLE_DIR / "pa_risk_by_procedure_payer.csv", index=False)
fig = px.bar(risk_by_proc, x="procedure_type", y="avg_hybrid_denial_risk", color="payer_type", title="Average PA Denial Risk by Procedure and Payer")
fig.show()
display(risk_by_proc)

,procedure_type,payer_type,avg_hybrid_denial_risk,high_risk_cases
18,post_acute,MAPD,0.701484,1
16,part_b_drug,MA_HMO,0.476028,1
11,SNF,MA_PPO,0.433242,2
4,DME,MA_HMO,0.341052,2
12,home_health,MAPD,0.302317,1
23,surgery,MA_PPO,0.252289,1
5,DME,MA_PPO,0.244650,0
17,part_b_drug,MA_PPO,0.192879,0
19,post_acute,MA_HMO,0.190489,0
8,MRI,MA_PPO,0.170315,0


## PA Decision

The model accuracy is useful for demonstration, but the correct business framing is risk prioritization. The output tells a team which authorization requests need stronger documentation before submission; it does not claim payer-specific production denial prediction.